# Solvers: Euler and Dopri5

`CompiledModel.run` accepts `solver=` — `"euler"` (fixed step) or a diffrax
method (`"dopri5"`, `"tsit5"`, `"heun"`, or a solver instance). Adaptive
stepping reports `result.solver.num_steps`; dense output exposes
`result.evaluate(t)`.


In [ ]:
import numpy as np

from summer4 import (
    Compartments,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)

state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], 0.3))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))
cm = model.compile()
y0 = PropertyData.wrap(pmap, np.array([999.0, 1.0, 0.0]))
plan = SavePlan(requests={"compartments": SaveRequest(Compartments())})


## Euler vs Dopri5 against the analytic S(t)

In [ ]:
# For this linear infection rate, S(t) = 999 * exp(-0.3 t).
ts = np.array([0.0, 1.0])
plan_ends = SavePlan(requests={"compartments": SaveRequest(Compartments(), ts=ts)})
euler = cm.run({}, y0, t0=0.0, steps=10_000, dt=1e-4, save=plan_ends, solver="euler")
dopri = cm.run(
    {}, y0, t0=0.0, t1=1.0, dt=0.1, save=plan_ends, solver="dopri5", rtol=1e-8, atol=1e-10
)
s_analytic = 999.0 * np.exp(-0.3)
np.testing.assert_allclose(float(np.asarray(dopri["compartments"].values.data)[-1, 0]), s_analytic, rtol=1e-5)
np.testing.assert_allclose(
    np.asarray(euler["compartments"].values.data)[-1],
    np.asarray(dopri["compartments"].values.data)[-1],
    rtol=1e-4,
    atol=1e-4,
)
assert dopri.solver is not None
assert int(dopri.solver.num_steps) != 10_000  # adaptive, not one step per Euler dt
assert int(dopri.solver.num_steps) < 10_000


## Off-grid save times use the solver interpolant

In [ ]:
t_off = 3.37
plan_off = SavePlan(
    requests={"compartments": SaveRequest(Compartments(), ts=np.array([0.0, t_off, 10.0]))}
)
res = cm.run(
    {}, y0, t0=0.0, t1=10.0, dt=1.0, save=plan_off, solver="dopri5", rtol=1e-8, atol=1e-10
)
np.testing.assert_allclose(np.asarray(res["compartments"].times.values), [0.0, t_off, 10.0])
s_off = float(np.asarray(res["compartments"].values.data)[1, 0])
np.testing.assert_allclose(s_off, 999.0 * np.exp(-0.3 * t_off), rtol=1e-5)


## Dense evaluation

In [ ]:
dense_plan = SavePlan(requests={"compartments": SaveRequest(Compartments())}, dense=True)
dense = cm.run(
    {}, y0, t0=0.0, t1=2.0, dt=0.1, save=dense_plan, solver="dopri5",
    rtol=1e-6, atol=1e-8, max_steps=512,
)
at0 = dense.evaluate(0.0)
np.testing.assert_allclose(np.asarray(at0.data), np.asarray(dense["compartments"].values.data)[0], rtol=1e-4)
assert dense.solver is not None and dense.solver.dense is True
print("euler steps (ref):", 10_000, "dopri5 steps:", int(dopri.solver.num_steps))
